# Showcase — Calidad del aire con datos SISAIRE/CAR reales

**Objetivo:** demostrar el ciclo estadístico completo (carga → calidad → descriptiva → cumplimiento → tendencia → pronóstico → reporte) consumiendo datos reales de la **Red de Monitoreo de Calidad del Aire** de la CAR vía `load_sisaire_local()`, sin duplicar archivos al repo.

**Datos:** archivos `CAR_<año>.csv` descargados manualmente del portal SISAIRE/IDEAM, almacenados en una carpeta externa apuntada por la variable de entorno `SISAIRE_LOCAL_DIR`. Si la variable no está configurada, el notebook cae a un dataset sintético equivalente (mismo esquema) para que sea reproducible.

**Contexto de dominio:** ver `docs/fuentes/calidad_aire.md` (ICA, normas, buenas prácticas metodológicas, RF/XGBoost como modelo de producción real).

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from estadistica_ambiental.io.connectors import load_sisaire_local
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.reporting.compliance_report import compliance_report

OUT_DIR = Path('data/output/showcase_sisaire')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Setup OK ->', OUT_DIR)

## 1. Carga — datos reales con fallback sintético

`load_sisaire_local(anios=, parametro=)` lee los `CAR_<año>.csv` de la carpeta indicada por `SISAIRE_LOCAL_DIR`. Si no está configurada, lanza `FileNotFoundError`; en ese caso generamos PM2.5 horario sintético con estacionalidad diaria + ruido.

In [ ]:
try:
    df = load_sisaire_local(anios=2024, parametro='pm25')
    fuente = 'SISAIRE/CAR real'
except FileNotFoundError as e:
    print(f'[fallback] {e}')
    rng = np.random.default_rng(42)
    n = 24 * 366
    fechas = pd.date_range('2024-01-01', periods=n, freq='h')
    diurno = 8 * np.sin(2 * np.pi * np.arange(n) / 24)
    estacional = 5 * np.sin(2 * np.pi * np.arange(n) / (24 * 365))
    pm25 = np.clip(22 + diurno + estacional + rng.normal(0, 6, n), 0, None)
    df = pd.DataFrame({'fecha': fechas, 'estacion': 'sintetica', 'pm25': pm25})
    fuente = 'sintetico (fallback)'

print(f'Fuente: {fuente}')
print(f'Registros: {len(df):,} | Estaciones: {df.estacion.nunique()} | '
      f'Periodo: {df.fecha.min()} -> {df.fecha.max()}')
df.head()

## 2. Calidad y validación

- `validate(df, linea_tematica='calidad_aire')` aplica los rangos físicos del dominio (ADR-006). NO elimina valores fuera de rango (ADR-002): los picos ambientales son señal real.
- `assess_quality(df, date_col='fecha')` reporta faltantes, gaps temporales, outliers estadísticos y congelamiento de sensor.

In [ ]:
val = validate(df, date_col='fecha', linea_tematica='calidad_aire')
print(f'Issues detectados: {val.has_issues()}')
print(f'Violaciones de rango (cols): {list(val.range_violations.keys())}')
print(f'Faltantes (>0): {val.missing}')


In [ ]:
calidad = assess_quality(df, date_col='fecha')
print(calidad.summary())

## 3. Resampling a diario

La normativa colombiana (Res. 2254/2017) y las guías OMS 2021 evalúan promedios **diarios** y **anuales**. Pasamos de horario a diario por estación.

In [ ]:
daily = (
    df.assign(dia=df['fecha'].dt.floor('D'))
      .groupby(['estacion', 'dia'], as_index=False)['pm25']
      .mean()
      .rename(columns={'dia': 'fecha'})
)
print(f'Promedios diarios: {len(daily):,} filas')
daily.head()

## 4. Descriptiva univariada

In [ ]:
desc = summarize(daily[['pm25']])
desc.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
for est, sub in daily.groupby('estacion'):
    ax.plot(sub['fecha'], sub['pm25'], lw=0.8, alpha=0.7, label=str(est))
ax.axhline(25, ls='--', c='orange', label='Norma CO anual (25 ug/m3)')
ax.axhline(15, ls='--', c='red', label='OMS 24h (15 ug/m3)')
ax.set_title('PM2.5 diario por estacion')
ax.set_ylabel('ug/m3'); ax.legend(loc='upper right', ncol=2, fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / 'pm25_diario.png', dpi=120)
plt.show()

## 5. Excedencias normativas

`exceedance_report(serie, variable='pm25')` cruza la serie contra:
- **Res. 2254/2017** (Colombia): 37 µg/m³ (24h), 25 µg/m³ (anual).
- **Guías OMS 2021**: 15 µg/m³ (24h), 5 µg/m³ (anual).

In [ ]:
rep = exceedance_report(daily['pm25'], variable='pm25')
rep

## 6. Tendencia — Mann-Kendall + Sen slope

Test no paramétrico, robusto a estacionalidad y distribuciones no normales. Estándar en hidrología y calidad del aire para series largas.

In [ ]:
serie_global = daily.groupby('fecha')['pm25'].mean()
mk = mann_kendall(serie_global)
for k in ('trend', 'pval', 'tau', 'slope'):
    print(f'{k:>6} = {mk[k]}')
sig = mk['pval'] < mk['alpha']
print(f"\n=> Tendencia {'SIGNIFICATIVA' if sig else 'no significativa'}: {mk['trend']}")

## 7. Pronóstico estacional ingenuo

Para no introducir aquí dependencias pesadas (statsmodels SARIMA), comparamos una predicción **naive estacional (lag-7 días)** contra los últimos 30 días observados. Para SARIMA/XGBoost real ver el notebook `notebooks/lineas_tematicas/.../calidad_aire.ipynb`.

In [ ]:
from estadistica_ambiental.evaluation.metrics import evaluate

serie = serie_global.dropna()
h = 30
y_true = serie.iloc[-h:].values
y_pred = serie.shift(7).iloc[-h:].values  # naive lag-7

metrics = evaluate(y_true, y_pred, domain='air_quality', pollutant='pm25')
for k, v in metrics.items():
    print(f'{k:>14} = {v:.3f}')

## 8. Reporte HTML de cumplimiento

`compliance_report` genera un HTML autoexplicativo con la tabla de excedencias y los gráficos clave — útil para entregar a un decisor sin necesidad de abrir el notebook.

In [ ]:
html_path = OUT_DIR / 'cumplimiento_pm25.html'
compliance_report(
    daily,
    variables=['pm25'],
    linea_tematica='calidad_aire',
    output=str(html_path),
)
print(f'Reporte: {html_path}')

---
## Cierre

Este notebook se puede ejecutar de extremo a extremo con dos modos:

- **Modo real:** `setx SISAIRE_LOCAL_DIR "<carpeta con CAR_*.csv>"` y reejecutar.
- **Modo demo:** sin variable de entorno, usa datos sintéticos equivalentes.

Las salidas (PNG, HTML) quedan en `data/output/showcase_sisaire/`. Para producir un dashboard real consumiendo estos resultados, crear un repo satélite que importe `estadistica_ambiental` como dependencia (ver README, sección *Consumir desde otro proyecto*).